# 5-Year Pipeline: Part 6.2 - Auto-Update GLOBAL Model
This notebook keeps your Global AI permanently up-to-date. 
1. It checks if there is any missing data up to yesterday and downloads it.
2. Triggers Notebooks 2 & 3 to update the calculated columns.
3. Triggers Notebooks 4.2 & 5.2 to retrain the Global High/Low Predictor on the new data and evaluate it.

In [ ]:
import os
import glob
import json
import datetime
import pandas as pd
import yfinance as yf

# Load Metadata
metadata_file = 'dataset_metadata.json'
if not os.path.exists(metadata_file):
    print(f"Error: {metadata_file} not found! Please ensure it is generated.")
else:
    with open(metadata_file, 'r') as f:
        metadata = json.load(f)
        
    last_date_str = metadata.get('last_updated', '2000-01-01')
    last_date = datetime.datetime.strptime(last_date_str, '%Y-%m-%d').date()
    yesterday = datetime.date.today() - datetime.timedelta(days=1)
    
    if last_date < yesterday:
        print(f"Data is out of date! Last date in dataset: {last_date}")
        print(f"Downloading ALL missing data from {last_date + datetime.timedelta(days=1)} to {yesterday}...")
        
        start_d = last_date + datetime.timedelta(days=1)
        end_d = datetime.date.today() # yfinance end date is exclusive
        
        # Download Nifty 50 delta
        nifty50 = yf.download("^NSEI", start=start_d, end=end_d, auto_adjust=False, progress=False)
        if isinstance(nifty50.columns, pd.MultiIndex): nifty50.columns = nifty50.columns.droplevel(1)
        
        nifty_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
        for c in nifty_cols:
            if c not in nifty50.columns: nifty50[c] = 0
        nifty50 = nifty50[nifty_cols]
        nifty50.rename(columns={'Open': 'Nifty50_Open', 'High': 'Nifty50_High', 'Low': 'Nifty50_Low', 'Close': 'Nifty50_Close', 'Volume': 'Nifty50_Volume'}, inplace=True)
        
        raw_dir = '5_year_data/raw'
        files = glob.glob(os.path.join(raw_dir, '*_data.csv'))
        
        updated_max_date = last_date_str
        
        for f_path in files:
            symbol = os.path.basename(f_path).replace('_data.csv', '')
            try:
                df_new = yf.download(f"{symbol}.NS", start=start_d, end=end_d, auto_adjust=False, progress=False)
                if df_new.empty: continue
                if isinstance(df_new.columns, pd.MultiIndex): df_new.columns = df_new.columns.droplevel(1)
                
                # Formatting
                df_new['Stock Symbol'] = symbol
                
                # We need Sector info from existing file quickly
                df_old = pd.read_csv(f_path, nrows=1)
                sector = df_old['Sector'].iloc[0]
                sector_idx = df_old['Sector Index'].iloc[0]
                
                df_new['Sector'] = sector
                df_new['Sector Index'] = sector_idx
                df_new['Trading_Day_Of_Week'] = df_new.index.day_name()
                if 'Adj Close' not in df_new.columns: df_new['Adj Close'] = df_new['Close'] if 'Close' in df_new.columns else None
                
                df_new = df_new.join(nifty50, how='left')
                
                # Align columns
                cols_needed = ['Stock Symbol', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'Sector', 'Sector Index', 'Nifty50_Open', 'Nifty50_High', 'Nifty50_Low', 'Nifty50_Close', 'Nifty50_Volume', 'Trading_Day_Of_Week']
                for c in cols_needed:
                    if c not in df_new.columns: df_new[c] = None
                df_new = df_new[cols_needed]
                
                # Append and save
                df_new.index.name = 'Date'
                df_new.to_csv(f_path, mode='a', header=False)
                print(f"Updated {symbol}")
                
                # update metadata for this symbol
                new_max = df_new.index.max().strftime('%Y-%m-%d')
                if symbol in metadata['symbols']:
                    metadata['symbols'][symbol]['last_date'] = new_max
                else:
                    metadata['symbols'][symbol] = {'last_date': new_max}
                    
                if new_max > updated_max_date:
                    updated_max_date = new_max
                    
            except Exception as e:
                print(f"Error updating {symbol}: {e}")
        
        # Update JSON Log
        metadata['last_updated'] = updated_max_date
        with open(metadata_file, 'w') as f:
            json.dump(metadata, f, indent=4)
            
        print(f"\n✅ All raw data successfully updated! JSON Log updated to {updated_max_date}.")
    else:
        print("✅ Raw Data is already up to date! No downloading needed.")


### Trigger the Data Processing & AI Retraining Pipeline
*(This safely runs the other notebooks in the background to recalculate indicators and retrain the AI)*

In [ ]:
print("1. Updating Calculated Columns (Running Notebook 2)...")
!jupyter nbconvert --execute --inplace 5_years_2_feature_generation.ipynb

print("2. Cleaning and Rebuilding Final Dataset (Running Notebook 3)...")
!jupyter nbconvert --execute --inplace 5_years_3_data_cleaning.ipynb

print("3. Retraining Global High/Low Models (Running Notebook 4.2)...")
!jupyter nbconvert --execute --inplace 5_years_4.2_global_model_training.ipynb


print("4. Generating New Inference & Evaluation Metrics (Running Notebook 5.2)...")
!jupyter nbconvert --execute --inplace 5_years_5.2_global_inference.ipynb

print("\n✅ GLOBAL MODEL UPDATE COMPLETE! Open Notebook 5.2 to see the new metrics!")
